# 04 — Consumer artifact bundle

The **Consumer MVP Bundle** is a versioned contract for thin apps (catalog viewer,
lineage explorer) to read pre-emitted JSON without reimplementing policy or traversal.

Two committed bundles serve different demo purposes:

- **`minimal`** (lineage-demo) — column lineage, warehouse bind warnings, impact JSON
- **`lineage-demo`** (sql_folder) — non-empty column impact trace

```text
cm compile / cm impact → bundle.manifest.json → vanilla viewer
```

See also: [examples/consumers/README.md](../consumers/README.md)

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _paths import (
    build_bundle_script,
    consumer_bundle_dir,
    consumer_scenario,
    repo_root,
)

BUNDLE_DIR = consumer_bundle_dir()
SCENARIO = consumer_scenario()
BUILD_SCRIPT = build_bundle_script()

print(f"bundle: {BUNDLE_DIR}")

## Load and validate the committed bundle

All schema validation is centralized in `clearmetric.core.validate`.

In [ ]:
from clearmetric.core.validate import (
    load_artifact_file,
    load_bundle_manifest_file,
    load_impact_output_file,
)

manifest = load_bundle_manifest_file(BUNDLE_DIR / "bundle.manifest.json")
print(json.dumps({k: manifest[k] for k in ("scenario_id", "label", "defaults")}, indent=2))

graph = load_artifact_file(BUNDLE_DIR / manifest["artifacts"]["graph"]["path"])
catalog = load_artifact_file(BUNDLE_DIR / manifest["artifacts"]["catalog"]["path"])
print(f"graph nodes: {len(graph.nodes)}  catalog nodes: {len(catalog.nodes)}")

In [ ]:
impact_key = manifest["defaults"]["impact_key"]
impact_ref = manifest["artifacts"]["impacts"][impact_key]
impact = load_impact_output_file(BUNDLE_DIR / impact_ref["path"])

print(f"impact: {impact_key}")
print(f"  selection_id: {impact['selection_id']}")
print(f"  related_ids: {len(impact['related_ids'])}")
print(f"  warnings: {[w['code'] for w in impact['warnings']]}")

## Lineage-demo bundle (non-empty impact)

The lineage explorer default bundle uses the sql_folder fixture.

In [ ]:
LINEAGE_BUNDLE_DIR = consumer_bundle_dir("lineage-demo")
lineage_manifest = load_bundle_manifest_file(LINEAGE_BUNDLE_DIR / "bundle.manifest.json")
lineage_key = lineage_manifest["defaults"]["impact_key"]
lineage_ref = lineage_manifest["artifacts"]["impacts"][lineage_key]
lineage_impact = load_impact_output_file(LINEAGE_BUNDLE_DIR / lineage_ref["path"])

print(f"lineage impact: {lineage_key}")
print(f"  related_ids ({len(lineage_impact['related_ids'])}):")
for node_id in lineage_impact["related_ids"]:
    print(f"    {node_id}")
assert len(lineage_impact["related_ids"]) >= 2

## Regenerate the bundle from the scenario recipe

`build_bundle.py` runs `cm compile` / `cm impact`, validates each artifact, and writes
`bundle.manifest.json` + `meta.json`.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    out = Path(tmp) / "bundle"
    result = subprocess.run(
        [sys.executable, str(BUILD_SCRIPT), "--scenario", str(SCENARIO), "--out", str(out)],
        capture_output=True,
        text=True,
        cwd=str(repo_root()),
    )
    assert result.returncode == 0, result.stderr or result.stdout
    print(result.stdout.strip())

    rebuilt = load_bundle_manifest_file(out / "bundle.manifest.json")
    assert rebuilt["scenario_id"] == "minimal"
    print("rebuilt bundle validates OK")

## Corpus checks (`checks.yaml`)

Hand-verified expectations live beside each scenario. CI runs them via
`tests/consumers/test_corpus_checks.py` — the notebook shows the same assertions inline.

In [ ]:
import yaml

checks = yaml.safe_load((SCENARIO / "checks.yaml").read_text())
for case in checks["cases"]:
    print(f"- {case['id']}")

catalog_kinds = {node.kind for node in catalog.nodes}
assert "column" in catalog_kinds and "table" in catalog_kinds
assert any(node.id == "column:orders.amount" for node in catalog.nodes)
assert impact["selection_id"] == "column:orders.amount"
print("minimal corpus checks satisfied on committed bundle")

## Open the HTML viewers

From the repo root:

```bash
python -m http.server 8000 --directory examples/consumers
# catalog-viewer/index.html?bundle=../bundles/minimal
# lineage-explorer/index.html?bundle=../bundles/lineage-demo
```